# **Chunking Method Development**

This notebook tunes five predefined chunking methods using development data only:

1. Sentence Splitter
2. Token Splitter
3. Sentence Window Node Parser
4. Semantic Splitter Node Parser
5. Hierarchical Node Parser

The evaluation and test benchmarks must not be used for parameter tuning.
After tuning, one configuration per method will be frozen for evaluation.

In [67]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyTorch: 2.7.1+cu126
CUDA available: True
GPU: NVIDIA L40S


In [66]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

In [68]:
from importlib.metadata import version
for package in ["llama-index", "llama-index-core", "llama-index-embeddings-huggingface", "transformers", "sentence-transformers", "huggingface-hub"]:
    print(package, version(package))

llama-index 0.14.23
llama-index-core 0.14.23
llama-index-embeddings-huggingface 0.7.0
transformers 4.46.3
sentence-transformers 5.7.0
huggingface-hub 0.26.2


In [69]:
from pathlib import Path
PROJECT_ROOT = Path.home() / "esg_rag_project"
RESULTS_PATH = PROJECT_ROOT / "outputs/qa_benchmark/development_chunking_results.jsonl"
print("Project:", PROJECT_ROOT.resolve(), PROJECT_ROOT.exists())
print("Results:", RESULTS_PATH.exists(), RESULTS_PATH.stat().st_size if RESULTS_PATH.exists() else 0)

Project: /data/home/zll/xh0862/esg_rag_project True
Results: True 832272


In [6]:
# %pip install llama-index

Defaulting to user installation because normal site-packages is not writeable
  Using cached llama_index-0.14.24-py3-none-any.whl.metadata (14 kB)
  Using cached llama_index_core-0.14.24-py3-none-any.whl.metadata (2.6 kB)
  Using cached llama_index_embeddings_openai-0.6.0-py3-none-any.whl.metadata (401 bytes)
  Using cached llama_index_llms_openai-0.7.10-py3-none-any.whl.metadata (3.0 kB)
  Using cached aiosqlite-0.22.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached banks-2.5.0-py3-none-any.whl.metadata (12 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached deprecated-1.3.1-py2.py3-none-any.whl.metadata (5.9 kB)
  Using cached dirtyjson-1.0.8-py3-none-any.whl.metadata (11 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached llama_index_workflows-2.23.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached tinytag-2.3.1-py3-none-any.whl.metadata (24 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata

In [20]:
# %pip install "llama-index==0.14.23" "llama-index-core==0.14.23" "llama-index-embeddings-huggingface==0.7.0" "transformers==4.46.3" "sentence-transformers==5.7.0" "huggingface-hub==0.26.2"

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 53.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 38.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 28.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.30.0
    Uninstalling huggingface_hub-1.30.0:
      Successfully uninstalled huggingface_hub-1.30.0
  Attempting uninstall: tokenizers━━━━━━━━━━━━━━ 0/7 [huggingface-hub]
    Found existing installation: tokenizers 0.23.20/7 [huggingface-hub]
    Uninstalling tokenizers-0.23.2:━━━━━━━━━ 0/7 [huggingface-hub]
      Successfully uninstalled tokenizers-0.23.2 0/7 [huggingface-hub]
  Attempting uninstall: transformers━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [tokenizers]
    Found existing installation: trans

#### a. Preparing datasets

In [70]:
# Import libraries and define project folders

from pathlib import Path
import json
import time
import pickle

import numpy as np
import pandas as pd

import hashlib
import os
from pathlib import Path

import copy
from datetime import datetime
from importlib.metadata import version

from llama_index.core import StorageContext, VectorStoreIndex, load_index_from_storage
from llama_index.core.schema import QueryBundle
from llama_index.core.schema import MetadataMode


import warnings
warnings.filterwarnings("ignore")  # suppress all warnings

In [71]:
# from importlib.metadata import version
# print(version("llama-index-core"))

In [72]:
# LlamaIndex components used by the five methods
from llama_index.core import (
    Document,
    Settings,
    StorageContext,
    VectorStoreIndex,
)

from llama_index.core.node_parser import (
    SentenceSplitter,
    TokenTextSplitter,
    SentenceWindowNodeParser,
    SemanticSplitterNodeParser,
    HierarchicalNodeParser,
    get_leaf_nodes,
)

from llama_index.core.postprocessor import (
    MetadataReplacementPostProcessor,
)

from llama_index.core.retrievers import (
    AutoMergingRetriever,
)

from llama_index.core.schema import MetadataMode
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.postprocessor import MetadataReplacementPostProcessor
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import AutoMergingRetriever

In [73]:
# Reuse the existing project and benchmark folders

PROJECT_ROOT = Path.home() / "esg_rag_project"
PARSED_FOLDER = PROJECT_ROOT / "data" / "parsed"
DEVELOPMENT_TEXT_FOLDER = PROJECT_ROOT / "data" / "splits" / "development"
BENCHMARK_FOLDER = PROJECT_ROOT / "outputs" / "qa_benchmark"

# Confirm that the existing folders are available
assert PROJECT_ROOT.exists(), f"Project folder not found: {PROJECT_ROOT}"
assert PARSED_FOLDER.exists(), f"Parsed-data folder not found: {PARSED_FOLDER}"
assert DEVELOPMENT_TEXT_FOLDER.exists(), f"Development-text folder not found: {DEVELOPMENT_TEXT_FOLDER}"
assert BENCHMARK_FOLDER.exists(), f"Benchmark folder not found: {BENCHMARK_FOLDER}"

print("Project folder:", PROJECT_ROOT)
print("Benchmark/output folder:", BENCHMARK_FOLDER)

Project folder: /data/home/zll/xh0862/esg_rag_project
Benchmark/output folder: /data/home/zll/xh0862/esg_rag_project/outputs/qa_benchmark


In [74]:
CHUNKING_DESIGN_PATH = BENCHMARK_FOLDER / "chunking_experiment_design_2.4.json"

In [75]:
# load development questions and parser outputs 
# Load JSONL files while ignoring empty lines

def load_jsonl(path):
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


UNITS_PATH = PARSED_FOLDER / "development_units.jsonl"
PROVISIONS_PATH = PARSED_FOLDER / "development_provisions.jsonl"
DEVELOPMENT_QA_PATH = BENCHMARK_FOLDER / "development_qa_generated_2.3.5-v3-direct-esg.jsonl"

for path in [UNITS_PATH, PROVISIONS_PATH, DEVELOPMENT_QA_PATH]:
    assert path.exists(), f"Required file not found: {path}"

development_units_df = pd.DataFrame(load_jsonl(UNITS_PATH))
development_provisions_df = pd.DataFrame(load_jsonl(PROVISIONS_PATH))
development_qa_df = pd.DataFrame(load_jsonl(DEVELOPMENT_QA_PATH))

# Retain only usable QA pairs from the final development prompt
development_qa_df = development_qa_df[development_qa_df["usable"]].copy()

print("Development units:", len(development_units_df))
print("Development provisions:", len(development_provisions_df))
print("Usable development questions:", len(development_qa_df))

Development units: 7847
Development provisions: 64576
Usable development questions: 56


In [76]:
# Standardise Articles and provisions into one gold-evidence lookup

unit_gold_df = development_units_df.rename(columns={
    "unit_id": "evidence_id",
    "unit_text": "gold_evidence_text",
    "start_char": "gold_start_char",
    "end_char": "gold_end_char",
})

provision_gold_df = development_provisions_df.rename(columns={
    "provision_id": "evidence_id",
    "provision_text": "gold_evidence_text",
    "start_char": "gold_start_char",
    "end_char": "gold_end_char",
})

gold_columns = [
    "evidence_id",
    "doc_id",
    "cleaned_filename",
    "gold_evidence_text",
    "gold_start_char",
    "gold_end_char",
]

development_gold_df = pd.concat(
    [unit_gold_df[gold_columns], provision_gold_df[gold_columns]],
    ignore_index=True,
)

# Attach the approved passage and offsets to every development question
development_queries_df = development_qa_df.merge(
    development_gold_df,
    on=["evidence_id", "doc_id"],
    how="left",
    validate="one_to_one",
)

assert development_queries_df["qa_id"].is_unique
assert development_queries_df["gold_evidence_text"].notna().all()
assert development_queries_df["gold_start_char"].notna().all()
assert development_queries_df["gold_end_char"].notna().all()

print("Development queries with gold evidence:", len(development_queries_df))

Development queries with gold evidence: 56


In [77]:
# Build the development corpus manifest from every available parser table

manifest_sources = []

for dataframe_name, dataframe in [
    ("units", development_units_df),
    ("provisions", development_provisions_df),
]:
    required_columns = {"doc_id", "cleaned_filename"}

    if required_columns.issubset(dataframe.columns):
        source_manifest = dataframe[
            ["doc_id", "cleaned_filename"]
        ].dropna().copy()

        source_manifest["manifest_source"] = dataframe_name
        manifest_sources.append(source_manifest)


assert manifest_sources, (
    "No parser table contains both doc_id "
    "and cleaned_filename."
)

document_manifest_df = (
    pd.concat(
        manifest_sources,
        ignore_index=True,
    )
    .assign(
        doc_id=lambda frame:
            frame["doc_id"].astype(str).str.strip(),
        cleaned_filename=lambda frame:
            frame["cleaned_filename"].astype(str).str.strip(),
    )
    .drop_duplicates(
        subset=["doc_id"],
        keep="first",
    )
)

print(
    "Documents in combined parser manifest:",
    len(document_manifest_df),
)

Documents in combined parser manifest: 283


In [78]:
# Use every development document, including non-gold distractor content

document_manifest_df = development_units_df[
    ["doc_id", "cleaned_filename"]
].drop_duplicates()

assert document_manifest_df["doc_id"].is_unique

development_documents = []

for row in document_manifest_df.itertuples(index=False):
    text_path = DEVELOPMENT_TEXT_FOLDER / row.cleaned_filename
    assert text_path.exists(), f"Cleaned document not found: {text_path}"

    # Exclude identifiers from embeddings so filenames cannot influence retrieval
    document = Document(
        text=text_path.read_text(encoding="utf-8", errors="replace"),
        id_=row.doc_id,
        metadata={
            "doc_id": row.doc_id,
            "split": "development",
            "cleaned_filename": row.cleaned_filename,
        },
        excluded_embed_metadata_keys=["doc_id", "split", "cleaned_filename"],
        excluded_llm_metadata_keys=["doc_id", "split", "cleaned_filename"],
    )

    development_documents.append(document)


# Confirm that every gold document exists in the retrieval corpus
corpus_doc_ids = {document.doc_id for document in development_documents}
gold_doc_ids = set(development_queries_df["doc_id"])

assert gold_doc_ids <= corpus_doc_ids

print("Development corpus documents:", len(development_documents))
print("Gold documents:", len(gold_doc_ids))
print("Development corpus loaded.")

Development corpus documents: 283
Gold documents: 44
Development corpus loaded.


#### b. Freeze the chunking methods and evaluation rules

In [79]:
# Adapt the paper's parameter grids to the current LlamaIndex implementation 
fixed_grid = [{"chunk_size": 128, "chunk_overlap": 20},
              {"chunk_size": 256, "chunk_overlap": 30},
              {"chunk_size": 512, "chunk_overlap": 50},
              {"chunk_size": 1024, "chunk_overlap": 100},
              {"chunk_size": 2048, "chunk_overlap": 200},]

In [80]:
CHUNKING_METHODS = {
    "sentence": {
        "parser": "SentenceSplitter",
        "parameter_grid": fixed_grid,
    },
    "token": {
        "parser": "TokenTextSplitter",
        "parameter_grid": fixed_grid,
    },
    "sentence_window": {
        "parser": "SentenceWindowNodeParser",
        "retrieval_unit": "sentence",
        "returned_context": "expanded_window",
        "parameter_grid": [{"window_size": size} for size in [1, 2, 3, 5, 7]],
    },

    "semantic": {
        "parser": "SemanticSplitterNodeParser",
        "boundary_embedding": "same_as_retrieval",
        "parameter_grid": [
            {"buffer_size": 1, "threshold": 90},
            {"buffer_size": 1, "threshold": 95},
            {"buffer_size": 1, "threshold": 98},
            {"buffer_size": 2, "threshold": 95},
            {"buffer_size": 3, "threshold": 95},
        ],
    },
    "hierarchical": {
        "parser": "HierarchicalNodeParser",
        "indexed_nodes": "leaf",
        "retriever": "AutoMergingRetriever",
        "auto_merging_threshold": 0.5,
        "parameter_grid": [
            {"chunk_sizes": [4096, 1024, 256]},
            {"chunk_sizes": [2048, 512, 128]},
            {"chunk_sizes": [2048, 1024, 512]},
            {"chunk_sizes": [1024, 512, 256]},
            {"chunk_sizes": [1024, 256, 64]},
        ],
    },
}

In [81]:
# Fixed comparison rules prevent larger returned contexts from gaining an unreported advantage
RETRIEVAL_K_VALUES = [1, 3, 5, 10]
FIXED_CONTEXT_TOKEN_BUDGET = 1000

RELEVANCE_RULES = {
    "complete": "Retrieved spans cumulatively cover the complete gold span in the correct document.",
    "partial": "At least one retrieved span overlaps the gold span in the correct document.",
}

SELECTION_RULE = [
    "highest complete Recall@5",
    "paired bootstrap comparison",
    "higher MRR",
    "higher Recall@10",
    "fewer retrieved tokens",
    "lower median retrieval latency",
]


In [82]:
# Save the design before development tuning begins
experiment_design = {
    "methods": CHUNKING_METHODS,
    "k_values": RETRIEVAL_K_VALUES,
    "token_budget": FIXED_CONTEXT_TOKEN_BUDGET,
    "relevance": RELEVANCE_RULES,
    "selection_rule": SELECTION_RULE,
}

CHUNKING_DESIGN_PATH = BENCHMARK_FOLDER / "chunking_experiment_design_2.4.json"
CHUNKING_DESIGN_PATH.write_text(json.dumps(experiment_design, indent=2, ensure_ascii=False), encoding="utf-8")


3238

In [83]:
# Confirm that the experiment contains five methods and 25 configurations
configuration_count = sum(len(method["parameter_grid"]) for method in CHUNKING_METHODS.values())

assert len(CHUNKING_METHODS) == 5
assert configuration_count == 25

print("Methods:", list(CHUNKING_METHODS))
print("Development configurations:", configuration_count)
print("Design saved:", CHUNKING_DESIGN_PATH)

Methods: ['sentence', 'token', 'sentence_window', 'semantic', 'hierarchical']
Development configurations: 25
Design saved: /data/home/zll/xh0862/esg_rag_project/outputs/qa_benchmark/chunking_experiment_design_2.4.json


### Embedding 

In [84]:
# Install the local BGE integration if it is not already available
# %pip install -q llama-index-embeddings-huggingface sentence-transformers

In [85]:
from importlib.metadata import version
for package in ["transformers", "sentence-transformers", "llama-index-embeddings-huggingface", "huggingface-hub"]:
    try: print(package, version(package))
    except: print(package, "not installed")

transformers 4.46.3
sentence-transformers 5.7.0
llama-index-embeddings-huggingface 0.7.0
huggingface-hub 0.26.2


In [86]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print("Import successful — no restart required.")

Import successful — no restart required.


In [87]:
# Freeze BGE-M3 as the common embedding model
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"

In [88]:
# Freeze BGE-M3 as the common embedding model
# Progress widgets are disabled because durable batch checkpoints report progress.
embed_model = HuggingFaceEmbedding(
    model_name=EMBEDDING_MODEL_NAME,
    max_length=8192,
    normalize=True,
    embed_batch_size=4,
    show_progress_bar=False,
)


In [89]:
Settings.embed_model = embed_model

In [90]:
# Confirm that the local model loads and returns consistent vectors
query_vector = embed_model.get_query_embedding("What environmental obligations apply?")
passage_vector = embed_model.get_text_embedding("The operator shall comply with environmental requirements.")

assert len(query_vector) == len(passage_vector) == 1024

In [91]:
# Record the exact embedding configuration for reproducibility
experiment_design["embedding_provider"] = "Hugging Face"
experiment_design["embedding_model"] = EMBEDDING_MODEL_NAME
experiment_design["embedding_mode"] = "dense"
experiment_design["embedding_max_length"] = 8192
experiment_design["embedding_dimensions"] = 1024
experiment_design["normalize_embeddings"] = True
experiment_design["query_instruction"] = None

CHUNKING_DESIGN_PATH.write_text(json.dumps(experiment_design, indent=2, ensure_ascii=False), encoding="utf-8")

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Embedding dimensions:", len(query_vector))
print("Maximum input length: 8192 tokens")
print("Embedding configuration frozen.")

Embedding model: BAAI/bge-m3
Embedding dimensions: 1024
Maximum input length: 8192 tokens
Embedding configuration frozen.


In [92]:
# Define one consistent node-creation function

def create_nodes(method, parameters, documents):
    """Create nodes using one frozen chunking configuration."""

    if method == "sentence":
        parser = SentenceSplitter(**parameters)
        return parser.get_nodes_from_documents(documents), None

    if method == "token":
        parser = TokenTextSplitter(**parameters)
        return parser.get_nodes_from_documents(documents), None

    if method == "sentence_window":
        parser = SentenceWindowNodeParser.from_defaults(
            window_size=parameters["window_size"],
            window_metadata_key="window",
            original_text_metadata_key="original_sentence",
        )
        return parser.get_nodes_from_documents(documents), None

    if method == "semantic":
        parser = SemanticSplitterNodeParser(
            embed_model=embed_model,
            buffer_size=parameters["buffer_size"],
            breakpoint_percentile_threshold=parameters["threshold"],
        )
        return parser.get_nodes_from_documents(documents), None

    if method == "hierarchical":
        parser = HierarchicalNodeParser.from_defaults(chunk_sizes=parameters["chunk_sizes"])
        all_nodes = parser.get_nodes_from_documents(documents)
        return get_leaf_nodes(all_nodes), all_nodes

    raise ValueError(f"Unknown chunking method: {method}")

In [93]:
# Define token counting and optional technical pilots
# The pilots already passed. Keep False during normal/resumed experiment runs.
from transformers import AutoTokenizer

RUN_TECHNICAL_PILOTS = False
bge_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)

PILOT_CONFIGS = {
    "sentence": {"chunk_size": 2048, "chunk_overlap": 200},
    "token": {"chunk_size": 2048, "chunk_overlap": 200},
    "sentence_window": {"window_size": 7},
    "semantic": {"buffer_size": 3, "threshold": 95},
    "hierarchical": {"chunk_sizes": [4096, 1024, 256]},
}

def count_bge_tokens(text):
    """Count BGE-M3 tokens without truncation."""
    return len(
        bge_tokenizer.encode(
            str(text),
            add_special_tokens=True,
            truncation=False,
        )
    )

if RUN_TECHNICAL_PILOTS:
    pilot_document = max(
        development_documents,
        key=lambda document: len(document.text),
    )
    pilot_results = []

    for method, parameters in PILOT_CONFIGS.items():
        indexed_nodes, all_nodes = create_nodes(
            method,
            parameters,
            [pilot_document],
        )

        assert indexed_nodes, f"{method} created no indexed nodes."
        assert len({node.node_id for node in indexed_nodes}) == len(indexed_nodes)
        assert all(node.metadata.get("doc_id") for node in indexed_nodes)

        embedded_texts = [
            node.get_content(metadata_mode=MetadataMode.NONE)
            for node in indexed_nodes
        ]

        if method == "sentence_window":
            assert all("window" in node.metadata for node in indexed_nodes)
            returned_texts = [node.metadata["window"] for node in indexed_nodes]
        elif method == "hierarchical":
            assert all(node.parent_node is not None for node in indexed_nodes)
            returned_texts = [
                node.get_content(metadata_mode=MetadataMode.NONE)
                for node in all_nodes
            ]
        else:
            returned_texts = embedded_texts

        embedded_lengths = [count_bge_tokens(text) for text in embedded_texts]
        returned_lengths = [count_bge_tokens(text) for text in returned_texts]
        longest_text = embedded_texts[embedded_lengths.index(max(embedded_lengths))]
        test_embedding = embed_model.get_text_embedding(longest_text)

        pilot_results.append({
            "method": method,
            "indexed_nodes": len(indexed_nodes),
            "total_nodes": len(all_nodes or indexed_nodes),
            "max_embedded_tokens": max(embedded_lengths),
            "max_returned_tokens": max(returned_lengths),
            "offsets_available": all(
                node.start_char_idx is not None and node.end_char_idx is not None
                for node in indexed_nodes
            ),
            "doc_id_preserved": all(
                node.metadata.get("doc_id") is not None
                for node in indexed_nodes
            ),
            "embedding_dimension": len(test_embedding),
        })

    pilot_results_df = pd.DataFrame(pilot_results)
    assert pilot_results_df["max_embedded_tokens"].le(8192).all()
    assert pilot_results_df["offsets_available"].all()
    assert pilot_results_df["doc_id_preserved"].all()
    assert pilot_results_df["embedding_dimension"].eq(1024).all()
    display(pilot_results_df)
    print("Five-method node and embedding smoke test passed.")
else:
    print("Technical node/embedding pilot skipped (already validated).")


Technical node/embedding pilot skipped (already validated).


In [94]:
# Define retrieval relevance scoring

def merge_intervals(intervals):
    """Merge overlapping or directly adjacent character spans."""
    merged = []

    for start, end in sorted(intervals):
        if start >= end:
            continue

        if not merged or start > merged[-1][1]:
            merged.append([start, end])
        else:
            merged[-1][1] = max(merged[-1][1], end)

    return merged


def trim_span_whitespace(text, start, end):
    """Remove boundary whitespace without changing source offsets."""
    while start < end and text[start].isspace():
        start += 1

    while end > start and text[end - 1].isspace():
        end -= 1

    return start, end


def locate_returned_span(result, document_text):
    """Locate one returned context inside its complete source document."""
    returned_text = result.node.get_content(metadata_mode=MetadataMode.NONE)

    if not returned_text:
        return None

    # Prefer parser offsets when they still describe the returned context
    start_hint = result.node.start_char_idx

    if start_hint is not None:
        end_hint = start_hint + len(returned_text)

        if document_text[start_hint:end_hint] == returned_text:
            return start_hint, end_hint

    # Expanded windows and merged parents may require exact text search
    positions = []
    position = document_text.find(returned_text)

    while position != -1:
        positions.append(position)
        position = document_text.find(returned_text, position + 1)

    if not positions:
        return None

    # Resolve repeated text using the parser's original position when available
    hint = start_hint if start_hint is not None else 0
    start = min(positions, key=lambda value: abs(value - hint))

    return start, start + len(returned_text)


def score_retrieval(query_row, retrieved_nodes, document_text, k):
    """Measure cumulative gold-evidence coverage within the first k results."""
    assert k > 0, "k must be positive."

    gold_start = int(query_row["evidence_start_char"])
    gold_end = int(query_row["evidence_end_char"])
    gold_start, gold_end = trim_span_whitespace(document_text, gold_start, gold_end)

    assert 0 <= gold_start < gold_end <= len(document_text), (
        f"Invalid gold span for evidence {query_row['evidence_id']}."
    )

    gold_doc_id = str(query_row["doc_id"])
    overlap_spans = []
    first_overlap_rank = None
    unlocated_results = 0

    for rank, result in enumerate(retrieved_nodes[:k], start=1):
        if str(result.node.metadata.get("doc_id")) != gold_doc_id:
            continue

        span = locate_returned_span(result, document_text)

        if span is None:
            unlocated_results += 1
            continue

        start, end = span

        if start < gold_end and end > gold_start:
            overlap_spans.append((max(start, gold_start), min(end, gold_end)))

            if first_overlap_rank is None:
                first_overlap_rank = rank

    merged_spans = merge_intervals(overlap_spans)
    covered_characters = sum(end - start for start, end in merged_spans)
    gold_characters = gold_end - gold_start
    coverage_ratio = min(covered_characters / gold_characters, 1.0)

    return {
        "complete_coverage": coverage_ratio == 1.0,
        "partial_overlap": coverage_ratio > 0,
        "coverage_ratio": coverage_ratio,
        "first_overlap_rank": first_overlap_rank,
        "reciprocal_rank": 0 if first_overlap_rank is None else 1 / first_overlap_rank,
        "unlocated_results": unlocated_results,
    }

print("Retrieval relevance scoring functions loaded.")

Retrieval relevance scoring functions loaded.


In [95]:
# Define fixed-token-budget scoring

def truncate_to_token_budget(text, token_budget):
    """Return the exact source-text prefix allowed by the token budget."""
    encoded = bge_tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        return_offsets_mapping=True,
    )

    offsets = encoded["offset_mapping"]

    if len(offsets) <= token_budget:
        return text, len(offsets)

    character_end = offsets[token_budget - 1][1]
    return text[:character_end], token_budget


def locate_text_span(result, returned_text, document_text):
    """Locate a complete or budget-truncated returned context."""
    if not returned_text:
        return None

    start_hint = result.node.start_char_idx

    if start_hint is not None:
        end_hint = start_hint + len(returned_text)

        if document_text[start_hint:end_hint] == returned_text:
            return start_hint, end_hint

    positions = []
    position = document_text.find(returned_text)

    while position != -1:
        positions.append(position)
        position = document_text.find(returned_text, position + 1)

    if not positions:
        return None

    hint = start_hint if start_hint is not None else 0
    start = min(positions, key=lambda value: abs(value - hint))

    return start, start + len(returned_text)


def score_budget_retrieval(query_row, retrieved_nodes, document_text, token_budget=1000):
    """Measure cumulative gold coverage under a fixed returned-token budget."""
    gold_start = int(query_row["evidence_start_char"])
    gold_end = int(query_row["evidence_end_char"])
    gold_start, gold_end = trim_span_whitespace(document_text, gold_start, gold_end)
    gold_doc_id = str(query_row["doc_id"])

    remaining_tokens = token_budget
    overlap_spans = []
    first_overlap_rank = None
    unlocated_results = 0
    included_results = 0

    for rank, result in enumerate(retrieved_nodes, start=1):
        if remaining_tokens == 0:
            break

        returned_text = result.node.get_content(metadata_mode=MetadataMode.NONE)
        budgeted_text, used_tokens = truncate_to_token_budget(
            returned_text,
            remaining_tokens,
        )

        remaining_tokens -= used_tokens
        included_results += 1

        if str(result.node.metadata.get("doc_id")) != gold_doc_id:
            continue

        span = locate_text_span(result, budgeted_text, document_text)

        if span is None:
            unlocated_results += 1
            continue

        start, end = span

        if start < gold_end and end > gold_start:
            overlap_spans.append((max(start, gold_start), min(end, gold_end)))

            if first_overlap_rank is None:
                first_overlap_rank = rank

    merged_spans = merge_intervals(overlap_spans)
    covered_characters = sum(end - start for start, end in merged_spans)
    coverage_ratio = min(covered_characters / (gold_end - gold_start), 1.0)

    return {
        "budget_complete_coverage": coverage_ratio == 1.0,
        "budget_partial_overlap": coverage_ratio > 0,
        "budget_coverage_ratio": coverage_ratio,
        "budget_first_overlap_rank": first_overlap_rank,
        "budget_tokens_used": token_budget - remaining_tokens,
        "budget_results_used": included_results,
        "budget_unlocated_results": unlocated_results,
    }

print("Fixed 1,000-token-budget scoring loaded.")

Fixed 1,000-token-budget scoring loaded.


In [96]:
# Build/load persistent indexes with node and embedding checkpoints
EMBEDDING_CHECKPOINT_BATCH_SIZE = 64

def corpus_fingerprint(documents):
    """Identify the exact development corpus."""
    digest = hashlib.sha256()
    for document in sorted(documents, key=lambda item: str(item.doc_id)):
        digest.update(str(document.doc_id).encode("utf-8"))
        digest.update(document.text.encode("utf-8"))
    return digest.hexdigest()


DEVELOPMENT_CORPUS_HASH = corpus_fingerprint(development_documents)
CORPUS_TAG = DEVELOPMENT_CORPUS_HASH[:10]


def node_content_fingerprint(nodes, configuration_id):
    """Identify node content without unstable generated node IDs."""
    digest = hashlib.sha256()
    digest.update(str(configuration_id).encode("utf-8"))
    digest.update(DEVELOPMENT_CORPUS_HASH.encode("utf-8"))
    digest.update(EMBEDDING_MODEL_NAME.encode("utf-8"))

    for position, node in enumerate(nodes):
        digest.update(str(position).encode("utf-8"))
        digest.update(str(node.metadata.get("doc_id", "")).encode("utf-8"))
        digest.update(
            node.get_content(metadata_mode=MetadataMode.NONE).encode("utf-8")
        )

    return digest.hexdigest()


def load_or_create_nodes(configuration):
    """Reuse parsed nodes so restart does not repeat semantic splitting."""
    configuration_id = configuration["configuration_id"]
    method = configuration["method"]
    parameters = configuration["parameters"]

    cache_folder = BENCHMARK_FOLDER / "node_checkpoints"
    cache_folder.mkdir(parents=True, exist_ok=True)
    cache_path = cache_folder / f"{configuration_id}.pkl"

    expected_metadata = {
        "configuration_id": configuration_id,
        "method": method,
        "parameters": parameters,
        "corpus_hash": DEVELOPMENT_CORPUS_HASH,
        "experiment_version": EXPERIMENT_VERSION,
        "llama_index_core_version": version("llama-index-core"),
    }

    if cache_path.exists():
        with cache_path.open("rb") as file:
            payload = pickle.load(file)

        if payload.get("metadata") == expected_metadata:
            print("Loaded saved nodes:", configuration_id)
            return (
                payload["indexed_nodes"],
                payload["all_nodes"],
                cache_path,
                0.0,
            )

        print("Ignoring incompatible node checkpoint:", cache_path.name)

    chunking_start = time.perf_counter()
    indexed_nodes, all_nodes = create_nodes(
        method,
        parameters,
        development_documents,
    )

    for node in indexed_nodes:
        node.excluded_embed_metadata_keys = list(node.metadata)

    chunking_seconds = time.perf_counter() - chunking_start
    payload = {
        "metadata": expected_metadata,
        "indexed_nodes": indexed_nodes,
        "all_nodes": all_nodes,
    }

    temporary_path = cache_path.with_name(cache_path.name + ".temporary")
    with temporary_path.open("wb") as file:
        pickle.dump(payload, file, protocol=pickle.HIGHEST_PROTOCOL)
        file.flush()
        os.fsync(file.fileno())
    os.replace(temporary_path, cache_path)

    print("Created and saved nodes:", configuration_id)
    return indexed_nodes, all_nodes, cache_path, chunking_seconds


def save_embedding_checkpoint(checkpoint_path, nodes, fingerprint):
    """Atomically save all embeddings completed so far."""
    completed_positions = [
        position
        for position, node in enumerate(nodes)
        if node.embedding is not None
    ]
    if not completed_positions:
        return

    embeddings = np.asarray(
        [nodes[position].embedding for position in completed_positions],
        dtype=np.float32,
    )
    temporary_path = checkpoint_path.with_name(
        checkpoint_path.stem + ".temporary.npz"
    )

    # Uncompressed NPZ is faster to write and reload than compressed NPZ.
    np.savez(
        temporary_path,
        fingerprint=np.asarray([fingerprint], dtype=str),
        positions=np.asarray(completed_positions, dtype=np.int64),
        embeddings=embeddings,
    )
    os.replace(temporary_path, checkpoint_path)


def load_embedding_checkpoint(checkpoint_path, nodes, expected_fingerprint):
    """Restore embeddings that match the current node content."""
    if not checkpoint_path.exists():
        return 0

    with np.load(checkpoint_path, allow_pickle=False) as checkpoint:
        saved_fingerprint = str(checkpoint["fingerprint"][0])
        if saved_fingerprint != expected_fingerprint:
            print("Ignoring incompatible embedding checkpoint:", checkpoint_path.name)
            return 0

        restored = 0
        for position, embedding in zip(
            checkpoint["positions"],
            checkpoint["embeddings"],
        ):
            position = int(position)
            if 0 <= position < len(nodes):
                nodes[position].embedding = embedding.tolist()
                restored += 1

    return restored


def embed_nodes_restart_safe(nodes, configuration_id):
    """Embed only missing nodes and checkpoint every completed batch."""
    checkpoint_folder = BENCHMARK_FOLDER / "embedding_checkpoints"
    checkpoint_folder.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_folder / f"{configuration_id}.npz"
    fingerprint = node_content_fingerprint(nodes, configuration_id)

    restored = load_embedding_checkpoint(
        checkpoint_path,
        nodes,
        fingerprint,
    )
    pending_positions = [
        position
        for position, node in enumerate(nodes)
        if node.embedding is None
    ]

    print(f"Restored embeddings: {restored}/{len(nodes)}")
    print("Embeddings still required:", len(pending_positions))

    for batch_start in range(
        0,
        len(pending_positions),
        EMBEDDING_CHECKPOINT_BATCH_SIZE,
    ):
        batch_positions = pending_positions[
            batch_start:batch_start + EMBEDDING_CHECKPOINT_BATCH_SIZE
        ]
        batch_nodes = [nodes[position] for position in batch_positions]
        batch_texts = [
            node.get_content(metadata_mode=MetadataMode.NONE)
            for node in batch_nodes
        ]
        batch_embeddings = embed_model.get_text_embedding_batch(
            batch_texts,
            show_progress=False,
        )

        for node, embedding in zip(batch_nodes, batch_embeddings):
            node.embedding = embedding

        save_embedding_checkpoint(checkpoint_path, nodes, fingerprint)
        completed = sum(node.embedding is not None for node in nodes)
        print(f"Embedded and saved: {completed}/{len(nodes)}")

    return checkpoint_path


def build_or_load_index(configuration):
    """Load a verified index, or build it from restart-safe checkpoints."""
    configuration_id = configuration["configuration_id"]
    method = configuration["method"]
    parameters = configuration["parameters"]
    index_path = (
        BENCHMARK_FOLDER
        / f"development_index_{CORPUS_TAG}_{configuration_id}")    
    marker_path = index_path / "index_metadata.json"

    expected_metadata = {
        "configuration_id": configuration_id,
        "method": method,
        "parameters": parameters,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "corpus_hash": DEVELOPMENT_CORPUS_HASH,
        "experiment_version": EXPERIMENT_VERSION,
    }

    if marker_path.exists():
        metadata = json.loads(marker_path.read_text(encoding="utf-8"))
        for key, value in expected_metadata.items():
            assert metadata.get(key) == value, f"Saved index has incompatible {key}."

        storage_context = StorageContext.from_defaults(
            persist_dir=str(index_path)
        )
        index = load_index_from_storage(
            storage_context,
            embed_model=embed_model,
        )
        print("Loaded saved index without re-embedding:", configuration_id)
        return index, storage_context, metadata

    indexed_nodes, all_nodes, node_cache_path, chunking_seconds = (
        load_or_create_nodes(configuration)
    )
    embedding_checkpoint_path = embed_nodes_restart_safe(
        indexed_nodes,
        configuration_id,
    )

    storage_context = StorageContext.from_defaults()
    if method == "hierarchical":
        storage_context.docstore.add_documents(all_nodes)

    indexing_start = time.perf_counter()
    index = VectorStoreIndex(
        indexed_nodes,
        storage_context=storage_context,
        embed_model=embed_model,
        show_progress=False,
    )
    indexing_seconds = time.perf_counter() - indexing_start

    index_path.mkdir(parents=True, exist_ok=True)
    index.storage_context.persist(persist_dir=str(index_path))

    metadata = {
        **expected_metadata,
        "indexed_nodes": len(indexed_nodes),
        "total_nodes": len(all_nodes or indexed_nodes),
        "chunking_seconds": chunking_seconds,
        "indexing_seconds": indexing_seconds,
        "node_checkpoint_path": str(node_cache_path),
        "embedding_checkpoint_path": str(embedding_checkpoint_path)
        }

    temporary_marker = marker_path.with_name("index_metadata.temporary.json")
    temporary_marker.write_text(
        json.dumps(metadata, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    os.replace(temporary_marker, marker_path)

    print("Index created and saved:", configuration_id)
    return index, index.storage_context, metadata


In [97]:
# Create stable configuration IDs and checkpoints
import hashlib

configuration_records = []

for method, method_design in CHUNKING_METHODS.items():
    for parameters in method_design["parameter_grid"]:
        parameter_text = json.dumps(parameters, sort_keys=True)
        parameter_hash = hashlib.sha256(parameter_text.encode()).hexdigest()[:8]

        configuration_records.append({
            "configuration_id": f"{method}_{parameter_hash}",
            "method": method,
            "parameters": parameters,
            "parameters_json": parameter_text,
        })

development_configurations_df = pd.DataFrame(configuration_records)

assert len(development_configurations_df) == 25
assert development_configurations_df["configuration_id"].is_unique

# Keep all experiment files inside the existing QA benchmark folder
DEVELOPMENT_CONFIG_PATH = BENCHMARK_FOLDER / "development_chunking_configurations.csv"
DEVELOPMENT_RESULTS_PATH = (BENCHMARK_FOLDER/ f"development_chunking_results_{CORPUS_TAG}.jsonl")

development_configurations_df.drop(columns="parameters").to_csv(
    DEVELOPMENT_CONFIG_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Configurations:", len(development_configurations_df))
print("Configuration manifest:", DEVELOPMENT_CONFIG_PATH)
print("Results checkpoint:", DEVELOPMENT_RESULTS_PATH)

Configurations: 25
Configuration manifest: /data/home/zll/xh0862/esg_rag_project/outputs/qa_benchmark/development_chunking_configurations.csv
Results checkpoint: /data/home/zll/xh0862/esg_rag_project/outputs/qa_benchmark/development_chunking_results_4cfb01ee22.jsonl


In [98]:
# Detect configurations already completed in an earlier run

if DEVELOPMENT_RESULTS_PATH.exists():
    completed_results = load_jsonl(DEVELOPMENT_RESULTS_PATH)
else:
    completed_results = []

completed_configuration_ids = {
    result["configuration_id"]
    for result in completed_results
    if result.get("configuration_complete", False)
}

pending_configurations_df = development_configurations_df[
    ~development_configurations_df["configuration_id"].isin(
        completed_configuration_ids
    )
].copy()

print("Completed configurations:", len(completed_configuration_ids))
print("Remaining configurations:", len(pending_configurations_df))

Completed configurations: 25
Remaining configurations: 0


In [99]:
# runs the chunking configurations on the development corpus 
# to determine which parameters work best for each method

# Freeze retrieval controls before tuning

EXPERIMENT_VERSION = "2.4.1"
BUDGET_CANDIDATE_K = 50
AUTO_MERGING_THRESHOLD = 0.5

# Fifty candidates provide a frozen pool for filling the 1,000-token budget
experiment_design["experiment_version"] = EXPERIMENT_VERSION
experiment_design["budget_candidate_k"] = BUDGET_CANDIDATE_K
experiment_design["auto_merging_threshold"] = AUTO_MERGING_THRESHOLD
experiment_design["llama_index_core_version"] = version("llama-index-core")

CHUNKING_DESIGN_PATH.write_text(
    json.dumps(experiment_design, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Experiment version:", EXPERIMENT_VERSION)
print("Fixed k:", RETRIEVAL_K_VALUES)
print("Budget candidate k:", BUDGET_CANDIDATE_K)
print("Auto-merging threshold:", AUTO_MERGING_THRESHOLD)


Experiment version: 2.4.1
Fixed k: [1, 3, 5, 10]
Budget candidate k: 50
Auto-merging threshold: 0.5


In [100]:
# Retrieve independently without modifying stored index nodes
window_postprocessor = MetadataReplacementPostProcessor(target_metadata_key="window")

def retrieve_nodes(index, storage_context, method, question, top_k):
    """Retrieve at one depth without mutating indexed nodes."""
    base_retriever = index.as_retriever(similarity_top_k=top_k)

    if method == "hierarchical":
        retriever = AutoMergingRetriever(
            base_retriever,
            storage_context,
            simple_ratio_thresh=AUTO_MERGING_THRESHOLD,
            verbose=False,)
        return retriever.retrieve(question)

    retrieved_nodes = base_retriever.retrieve(question)

    # Metadata replacement modifies node content, so process copies only
    if method == "sentence_window":
        retrieved_nodes = copy.deepcopy(retrieved_nodes)
        retrieved_nodes = window_postprocessor.postprocess_nodes(
            retrieved_nodes,
            query_bundle=QueryBundle(question),
        )

    return retrieved_nodes


def audit_retrieved_nodes(retrieved_nodes):
    """Preserve rankings, provenance, spans and returned-context sizes."""
    audit_records = []

    for rank, result in enumerate(retrieved_nodes, start=1):
        doc_id = str(result.node.metadata.get("doc_id"))
        document_text = document_text_by_id.get(doc_id)
        span = locate_returned_span(result, document_text) if document_text else None
        returned_text = result.node.get_content(metadata_mode=MetadataMode.NONE)

        audit_records.append({
            "rank": rank,
            "node_id": result.node.node_id,
            "doc_id": doc_id,
            "score": result.score,
            "start_char": span[0] if span else None,
            "end_char": span[1] if span else None,
            "returned_tokens": count_bge_tokens(returned_text),
        })

    return audit_records

In [101]:
# Evaluate one configuration with restart-safe, versioned checkpoints

document_text_by_id = {
    str(document.doc_id): document.text
    for document in development_documents
}


def append_checkpoint(record):
    """Append and flush one complete record to durable storage."""
    with DEVELOPMENT_RESULTS_PATH.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")
        file.flush()
        os.fsync(file.fileno())


def matches_current_experiment(record, configuration_id):
    """Prevent reuse of results from changed data or methodology."""
    return (
        record.get("configuration_id") == configuration_id
        and record.get("experiment_version") == EXPERIMENT_VERSION
        and record.get("corpus_hash") == DEVELOPMENT_CORPUS_HASH
        and record.get("embedding_model") == EMBEDDING_MODEL_NAME
    )


def run_development_configuration(configuration):
    """Evaluate one configuration across all development questions."""
    configuration_id = configuration["configuration_id"]
    method = configuration["method"]

    existing_records = (
        load_jsonl(DEVELOPMENT_RESULTS_PATH)
        if DEVELOPMENT_RESULTS_PATH.exists()
        else []
    )

    # Do nothing when this exact experiment has already finished
    if any(
        matches_current_experiment(record, configuration_id)
        and record.get("configuration_complete", False)
        for record in existing_records
    ):
        print("Configuration already complete:", configuration_id)
        return

    completed_qa_ids = {
        record["qa_id"]
        for record in existing_records
        if record.get("record_type") == "query_result"
        and matches_current_experiment(record, configuration_id)
    }

    index, storage_context, index_metadata = build_or_load_index(configuration)

    pending_queries = development_queries_df[
        ~development_queries_df["qa_id"].isin(completed_qa_ids)
    ]

    print("Configuration:", configuration_id)
    print("Completed queries:", len(completed_qa_ids))
    print("Remaining queries:", len(pending_queries))

    for number, (_, query_row) in enumerate(pending_queries.iterrows(), start=1):
        document_text = document_text_by_id[str(query_row["doc_id"])]

        record = {
            "record_type": "query_result",
            "configuration_complete": False,
            "configuration_id": configuration_id,
            "method": method,
            "parameters": configuration["parameters"],
            "experiment_version": EXPERIMENT_VERSION,
            "corpus_hash": DEVELOPMENT_CORPUS_HASH,
            "embedding_model": EMBEDDING_MODEL_NAME,
            "qa_id": query_row["qa_id"],
            "evidence_id": query_row["evidence_id"],
            "doc_id": str(query_row["doc_id"]),
        }

        # Retrieve separately because hierarchical merging depends on k
        for k in RETRIEVAL_K_VALUES:
            retrieval_start = time.perf_counter()
            retrieved_nodes = retrieve_nodes(
                index,
                storage_context,
                method,
                query_row["question"],
                k,
            )
            retrieval_seconds = time.perf_counter() - retrieval_start
            scores = score_retrieval(query_row, retrieved_nodes, document_text, k)

            record[f"complete_recall_at_{k}"] = scores["complete_coverage"]
            record[f"partial_recall_at_{k}"] = scores["partial_overlap"]
            record[f"coverage_at_{k}"] = scores["coverage_ratio"]
            record[f"unlocated_at_{k}"] = scores["unlocated_results"]
            record[f"retrieval_seconds_at_{k}"] = retrieval_seconds
            record[f"retrieved_at_{k}"] = audit_retrieved_nodes(retrieved_nodes)

            if k == 10:
                record["reciprocal_rank_at_10"] = scores["reciprocal_rank"]

        # Retrieve from a larger frozen pool for fixed-budget evaluation
        budget_start = time.perf_counter()
        budget_nodes = retrieve_nodes(
            index,
            storage_context,
            method,
            query_row["question"],
            BUDGET_CANDIDATE_K,
        )
        record["budget_retrieval_seconds"] = time.perf_counter() - budget_start

        record.update(
            score_budget_retrieval(
                query_row,
                budget_nodes,
                document_text,
                FIXED_CONTEXT_TOKEN_BUDGET,
            )
        )
        record["budget_retrieved_nodes"] = audit_retrieved_nodes(budget_nodes)

        append_checkpoint(record)
        print(f"[{number}/{len(pending_queries)}] saved | {query_row['qa_id']}")

    # Mark completion only after all 60 query records are saved
    append_checkpoint({
        "record_type": "configuration_marker",
        "configuration_id": configuration_id,
        "configuration_complete": True,
        "experiment_version": EXPERIMENT_VERSION,
        "corpus_hash": DEVELOPMENT_CORPUS_HASH,
        "embedding_model": EMBEDDING_MODEL_NAME,
        **index_metadata,
    })

    print("Configuration completed:", configuration_id)

In [102]:
print("Corpus documents:", len(development_documents))
print("Corpus tag:", CORPUS_TAG)
print("Results path:", DEVELOPMENT_RESULTS_PATH)

Corpus documents: 283
Corpus tag: 4cfb01ee22
Results path: /data/home/zll/xh0862/esg_rag_project/outputs/qa_benchmark/development_chunking_results_4cfb01ee22.jsonl


In [103]:
document_text_by_id = {str(d.doc_id).strip(): d.text for d in development_documents}
development_queries_df["doc_id"] = development_queries_df["doc_id"].astype(str).str.strip()

missing = sorted(set(development_queries_df["doc_id"]) - set(document_text_by_id))

print("Corpus documents:", len(development_documents))
print("Query documents:", development_queries_df["doc_id"].nunique())
print("Missing:", missing)

assert len(development_documents) == 283
assert not missing

Corpus documents: 283
Query documents: 44
Missing: []


In [104]:
# Cache the 56 query embeddings for reuse across all remaining configurations

QUERY_EMBEDDING_PATH = BENCHMARK_FOLDER / "development_query_embeddings_bge_m3.npz"

if QUERY_EMBEDDING_PATH.exists():
    saved = np.load(QUERY_EMBEDDING_PATH, allow_pickle=False)
    query_embedding_by_question = dict(zip(saved["questions"], saved["embeddings"]))
    print("Loaded query embeddings:", len(query_embedding_by_question))

else:
    questions = development_queries_df["question"].astype(str).tolist()
    embeddings = [
        embed_model.get_query_embedding(question)
        for question in questions
    ]

    np.savez_compressed(
        QUERY_EMBEDDING_PATH,
        questions=np.array(questions),
        embeddings=np.asarray(embeddings, dtype=np.float32),
    )

    query_embedding_by_question = dict(zip(questions, embeddings))
    print("Saved query embeddings:", len(query_embedding_by_question))

assert len(query_embedding_by_question) == len(development_queries_df)

Loaded query embeddings: 56


In [105]:
# Retrieve without repeatedly embedding the same question

def retrieve_nodes(index, storage_context, method, question, top_k):
    query_bundle = QueryBundle(
        query_str=question,
        embedding=query_embedding_by_question[question],
    )

    base_retriever = index.as_retriever(similarity_top_k=top_k)

    if method == "hierarchical":
        retriever = AutoMergingRetriever(
            base_retriever,
            storage_context,
            simple_ratio_thresh=AUTO_MERGING_THRESHOLD,
            verbose=False,
        )
        return retriever.retrieve(query_bundle)

    retrieved_nodes = base_retriever.retrieve(query_bundle)

    if method == "sentence_window":
        retrieved_nodes = copy.deepcopy(retrieved_nodes)
        retrieved_nodes = window_postprocessor.postprocess_nodes(
            retrieved_nodes,
            query_bundle=query_bundle,
        )

    return retrieved_nodes

print("Cached-query retrieval enabled.")

Cached-query retrieval enabled.


In [106]:
print("Documents:", len(development_documents))
print("Queries:", len(development_queries_df))
print("Query embeddings:", len(query_embedding_by_question))
print("Corpus hash:", DEVELOPMENT_CORPUS_HASH)

assert len(development_documents) == 283
assert len(development_queries_df) == 56
assert len(query_embedding_by_question) == 56

Documents: 283
Queries: 56
Query embeddings: 56
Corpus hash: 4cfb01ee22d2a6d039d64b1dc58be32815ce601aabbc01134cfcc130acaaeb59


In [107]:
# Align development-query columns with the frozen retrieval scorer

all_development_qa_df = pd.DataFrame(load_jsonl(DEVELOPMENT_QA_PATH))

print("Generated development records:", len(all_development_qa_df))
print("Usable development records:", int(all_development_qa_df["usable"].sum()))

assert len(development_queries_df) == int(
    all_development_qa_df["usable"].sum()
), "The query table does not match the usable development subset."

development_queries_df = development_queries_df.copy()

development_queries_df["original_evidence_text"] = (
    development_queries_df["gold_evidence_text"]
)
development_queries_df["evidence_start_char"] = (
    pd.to_numeric(development_queries_df["gold_start_char"]).astype(int)
)
development_queries_df["evidence_end_char"] = (
    pd.to_numeric(development_queries_df["gold_end_char"]).astype(int)
)

print("Retrieval queries ready:", len(development_queries_df))

Generated development records: 60
Usable development records: 56
Retrieval queries ready: 56


In [108]:
# Verify every gold span against its complete source document

document_text_by_id = {
    str(document.doc_id): document.text
    for document in development_documents
}

offset_issues = []

for _, row in development_queries_df.iterrows():
    document_text = document_text_by_id[str(row["doc_id"])]
    start = int(row["evidence_start_char"])
    end = int(row["evidence_end_char"])
    extracted_text = document_text[start:end].strip()
    gold_text = str(row["original_evidence_text"]).strip()

    if extracted_text != gold_text:
        offset_issues.append({
            "qa_id": row["qa_id"],
            "evidence_id": row["evidence_id"],
            "extracted_text": extracted_text,
            "gold_text": gold_text,
        })

offset_issues_df = pd.DataFrame(offset_issues)

print("Validated gold spans:", len(development_queries_df) - len(offset_issues_df))
print("Offset issues:", len(offset_issues_df))

if not offset_issues_df.empty:
    display(offset_issues_df)

assert offset_issues_df.empty, "Resolve the displayed gold-offset issues."

Validated gold spans: 56
Offset issues: 0


In [109]:
# Standardize development-query columns required by retrieval scoring

column_aliases = {
    "gold_evidence_text": "original_evidence_text",
    "gold_start_char": "evidence_start_char",
    "gold_end_char": "evidence_end_char",
}

for source, target in column_aliases.items():
    if target not in development_queries_df.columns:
        development_queries_df[target] = development_queries_df[source]

required_columns = [
    "qa_id",
    "evidence_id",
    "doc_id",
    "question",
    "original_evidence_text",
    "evidence_start_char",
    "evidence_end_char",
]

assert development_queries_df[required_columns].notna().all().all()
assert development_queries_df["qa_id"].is_unique

print("Development queries ready:", len(development_queries_df))

Development queries ready: 56


In [ ]:
# first_configuration = development_configurations_df.iloc[0].to_dict()
# run_development_configuration(first_configuration)

Loaded saved index without re-embedding: sentence_3c2ed685
Configuration: sentence_3c2ed685
Completed queries: 0
Remaining queries: 56
[1/56] saved | 36_2016_TT-BTC_m_308617_article_0010_q01
[2/56] saved | 203_QD-TTg_m_532187_article_0002_q01
[3/56] saved | 60_2023_ND-CP_m_577202_article_0002_q01
[4/56] saved | 09_2026_TT-BNNMT_m_696445_article_0009_clause_001_q01
[5/56] saved | 31_2014_QD-TTg_m_228707_article_0005_clause_001_q01
[6/56] saved | 36_2020_ND-CP_m_440993_article_0020_clause_006_q01
[7/56] saved | 81_2023_QH15_m_565003_article_0012_clause_001_q01
[8/56] saved | 08_2025_TT-BNNMT_m_666356_article_0001_embedded_article_001_q01
[9/56] saved | 05_2025_ND-CP_642709_article_0001_embedded_article_016_q01
[10/56] saved | 146_2025_QH15_706223_article_0007_embedded_article_001_q01
[11/56] saved | 247_2025_QH15_m_687497_article_0001_embedded_article_001_q01
[12/56] saved | 30_TB-VPCP_m_507657_document_fallback_0001_lettered_item_001_q01
[13/56] saved | 24_NQ-TW_m_203397_roman_section_0

#### Audit the first completed development configuration

In [110]:

# Audit the first completed development configuration

development_results = load_jsonl(DEVELOPMENT_RESULTS_PATH)

# Keep only records from the current frozen experiment
current_results = [
    record for record in development_results
    if record.get("experiment_version") == EXPERIMENT_VERSION
    and record.get("corpus_hash") == DEVELOPMENT_CORPUS_HASH
    and record.get("embedding_model") == EMBEDDING_MODEL_NAME
]

completed_ids = {
    record["configuration_id"]
    for record in current_results
    if record.get("record_type") == "configuration_marker"
    and record.get("configuration_complete", False)
}

assert completed_ids, "No completed configuration was found."

# Audit the first completed configuration in manifest order
audit_id = next(
    config_id
    for config_id in development_configurations_df["configuration_id"]
    if config_id in completed_ids
)

audit_records = [
    record for record in current_results
    if record.get("record_type") == "query_result"
    and record.get("configuration_id") == audit_id
]

audit_df = pd.DataFrame(audit_records)

# Confirm complete and unique query coverage
expected_qa_ids = set(development_queries_df["qa_id"])
assert len(audit_df) == len(expected_qa_ids)
assert audit_df["qa_id"].is_unique
assert set(audit_df["qa_id"]) == expected_qa_ids

coverage_columns = [
    f"coverage_at_{k}" for k in RETRIEVAL_K_VALUES
] + ["budget_coverage_ratio"]

unlocated_columns = [
    f"unlocated_at_{k}" for k in RETRIEVAL_K_VALUES
] + ["budget_unlocated_results"]

required_columns = [
    "reciprocal_rank_at_10",
    "budget_complete_coverage",
    "budget_partial_overlap",
    "budget_tokens_used",
    "budget_retrieval_seconds",
    *coverage_columns,
    *unlocated_columns,
]

assert audit_df[required_columns].notna().all().all()
assert audit_df[coverage_columns].apply(
    lambda column: column.between(0, 1).all()
).all()
assert audit_df["reciprocal_rank_at_10"].between(0, 1).all()
assert audit_df["budget_tokens_used"].between(
    1, FIXED_CONTEXT_TOKEN_BUDGET
).all()

# Confirm Boolean recall fields agree with numeric coverage
for k in RETRIEVAL_K_VALUES:
    coverage = audit_df[f"coverage_at_{k}"]

    assert audit_df[f"complete_recall_at_{k}"].eq(
        coverage.eq(1)
    ).all()
    assert audit_df[f"partial_recall_at_{k}"].eq(
        coverage.gt(0)
    ).all()
    assert audit_df[f"retrieval_seconds_at_{k}"].ge(0).all()
    assert audit_df[f"retrieved_at_{k}"].apply(
        lambda value: isinstance(value, list)
    ).all()

budget_coverage = audit_df["budget_coverage_ratio"]

assert audit_df["budget_complete_coverage"].eq(
    budget_coverage.eq(1)
).all()
assert audit_df["budget_partial_overlap"].eq(
    budget_coverage.gt(0)
).all()
assert audit_df["budget_retrieved_nodes"].apply(
    lambda value: isinstance(value, list)
).all()

# Stop if any returned context could not be mapped to its source
unlocated_counts = audit_df[unlocated_columns].fillna(0)
unlocated_total = int(unlocated_counts.to_numpy().sum())

if unlocated_total:
    problem_rows = audit_df[unlocated_counts.sum(axis=1).gt(0)]
    display(problem_rows[["qa_id", "evidence_id", *unlocated_columns]])
    raise ValueError("Resolve provenance failures before continuing.")

# Report the principal audit metrics
summary = {
    "configuration_id": audit_id,
    "method": audit_df["method"].iloc[0],
    "queries": len(audit_df),
    "partial_mrr_at_10": audit_df["reciprocal_rank_at_10"].mean(),
    "budget_complete_recall": audit_df["budget_complete_coverage"].mean(),
    "budget_partial_recall": audit_df["budget_partial_overlap"].mean(),
    "median_budget_tokens": audit_df["budget_tokens_used"].median(),
}

for k in RETRIEVAL_K_VALUES:
    summary[f"complete_recall_at_{k}"] = audit_df[
        f"complete_recall_at_{k}"
    ].mean()
    summary[f"partial_recall_at_{k}"] = audit_df[
        f"partial_recall_at_{k}"
    ].mean()
    summary[f"median_latency_at_{k}"] = audit_df[
        f"retrieval_seconds_at_{k}"
    ].median()

display(pd.DataFrame.from_dict(summary, orient="index", columns=["value"]))

print("Unique query results:", len(audit_df))
print("Unlocated retrieval occurrences:", unlocated_total)
print("First development configuration passed the full audit.")

,value
configuration_id,sentence_3c2ed685
method,sentence
queries,56
partial_mrr_at_10,0.699554
budget_complete_recall,0.178571
budget_partial_recall,0.875
median_budget_tokens,1000.0
complete_recall_at_1,0.160714
partial_recall_at_1,0.625
median_latency_at_1,5.120697


Unique query results: 56
Unlocated retrieval occurrences: 0
First development configuration passed the full audit.


In [111]:
# Identify the next unfinished development configuration

development_results = load_jsonl(DEVELOPMENT_RESULTS_PATH)

completed_configuration_ids = {
    record["configuration_id"]
    for record in development_results
    if record.get("record_type") == "configuration_marker"
    and record.get("configuration_complete", False)
    and record.get("experiment_version") == EXPERIMENT_VERSION
    and record.get("corpus_hash") == DEVELOPMENT_CORPUS_HASH
    and record.get("embedding_model") == EMBEDDING_MODEL_NAME
}

pending_configurations_df = development_configurations_df[
    ~development_configurations_df["configuration_id"].isin(
        completed_configuration_ids
    )
].copy()

print("Completed configurations:", len(completed_configuration_ids))
print("Remaining configurations:", len(pending_configurations_df))
display(pending_configurations_df.head())

Completed configurations: 25
Remaining configurations: 0


,configuration_id,method,parameters,parameters_json


#### Run the remaining development configurations

Each remaining configuration is evaluated against the same 283-document corpus and 56 development queries. One configuration is run at a time so that node, embedding and query-level checkpoints can be recovered after interruption.

In [ ]:
# Run one unfinished configuration

# results = load_jsonl(DEVELOPMENT_RESULTS_PATH)

# completed_ids = {
    record["configuration_id"] for record in results
    if record.get("record_type") == "configuration_marker"
    and record.get("configuration_complete", False)
    and record.get("experiment_version") == EXPERIMENT_VERSION
    and record.get("corpus_hash") == DEVELOPMENT_CORPUS_HASH
    and record.get("embedding_model") == EMBEDDING_MODEL_NAME
}

pending_df = development_configurations_df[
    ~development_configurations_df["configuration_id"].isin(completed_ids)
]

print("Completed:", len(completed_ids), "| Remaining:", len(pending_df))

if not pending_df.empty:
    configuration = pending_df.iloc[0].to_dict()
    print("Running:", configuration["configuration_id"], configuration["parameters"])
    run_development_configuration(configuration)
else:
    print("All 25 configurations are complete.")

Completed: 24 | Remaining: 1
Running: hierarchical_521fae18 {'chunk_sizes': [1024, 256, 64]}
Created and saved nodes: hierarchical_521fae18
Restored embeddings: 0/92722
Embeddings still required: 92722
Embedded and saved: 64/92722
Embedded and saved: 128/92722
Embedded and saved: 192/92722
Embedded and saved: 256/92722
Embedded and saved: 320/92722
Embedded and saved: 384/92722
Embedded and saved: 448/92722
Embedded and saved: 512/92722
Embedded and saved: 576/92722
Embedded and saved: 640/92722
Embedded and saved: 704/92722
Embedded and saved: 768/92722
Embedded and saved: 832/92722
Embedded and saved: 896/92722
Embedded and saved: 960/92722
Embedded and saved: 1024/92722
Embedded and saved: 1088/92722
Embedded and saved: 1152/92722
Embedded and saved: 1216/92722
Embedded and saved: 1280/92722
Embedded and saved: 1344/92722
Embedded and saved: 1408/92722
Embedded and saved: 1472/92722
Embedded and saved: 1536/92722
Embedded and saved: 1600/92722
Embedded and saved: 1664/92722
Embedded

In [52]:
# Keep a note that sentence_window_742e7319 produced one 13204 > 8192 
# tokenizer warning. We can audit long contexts after the 
# primary configurations finish; do not change the processing logic midway, 
# because that would make later configurations inconsistent with earlier ones.

# window size 1

In [112]:
results = load_jsonl(DEVELOPMENT_RESULTS_PATH)
current = [r for r in results if r.get("experiment_version") == EXPERIMENT_VERSION and r.get("corpus_hash") == DEVELOPMENT_CORPUS_HASH and r.get("embedding_model") == EMBEDDING_MODEL_NAME]
markers = [r for r in current if r.get("record_type") == "configuration_marker" and r.get("configuration_complete", False)]
query_results = [r for r in current if r.get("record_type") == "query_result"]
counts = pd.Series([r["configuration_id"] for r in query_results]).value_counts()
assert len({r["configuration_id"] for r in markers}) == 25
assert len(query_results) == 25 * 56
assert len(counts) == 25 and counts.eq(56).all()
assert not pd.DataFrame(query_results).duplicated(["configuration_id", "qa_id"]).any()
print("Final audit passed: 25 configurations, 1,400 unique query results.")

Final audit passed: 25 configurations, 1,400 unique query results.
